# Notebook 11: DQN Curriculum Experiments

Este notebook cria uma linha experimental separada para `DQN` com curriculum de oponentes, reward shaping por licao e self-play com pool de snapshots.


## Objetivo

O objetivo deste notebook e:

- comparar varias agendas de curriculum para `DQN`;
- medir a influencia conjunta de reward shaping e ordem de oponentes;
- manter o notebook base `03_dqn_self_play.ipynb` como referencia separada.


In [ ]:
from __future__ import annotations

import copy
import json
import statistics
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import torch

ROOT = Path.cwd().resolve()
while not (ROOT / "config.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "config.yaml").exists():
    raise FileNotFoundError("Could not find project root containing config.yaml")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

OUTPUTS = ROOT / "outputs"
CURRICULUM_OUTPUTS = OUTPUTS / "dqn_curriculum_experiments"
CURRICULUM_OUTPUTS.mkdir(parents=True, exist_ok=True)

from connect4_rl.config import load_config
from connect4_rl.experiments.dqn_curriculum import (
    build_default_dqn_curricula,
    expand_curriculum_schedule,
    train_dqn_with_curriculum,
)

CONFIG = load_config(ROOT / "config.yaml")
NOTEBOOK_DEVICE = CONFIG.resolve_device()
if NOTEBOOK_DEVICE == "cuda":
    torch.set_default_device(NOTEBOOK_DEVICE)
print({"device": NOTEBOOK_DEVICE, "outputs": str(CURRICULUM_OUTPUTS)})


## Passo 1: Configuracao da experiencia


In [ ]:
run_training = False

curricula = build_default_dqn_curricula()
selected_agendas = [
    "curriculum_classic",
    "curriculum_self_bridge",
    "curriculum_probabilistic_mix",
]
seeds = [7, 17, 27]

episodes = 180
eval_interval = 30
eval_games = 12

experiment_config = copy.deepcopy(CONFIG)
experiment_config.global_.device = NOTEBOOK_DEVICE
experiment_config.dqn.episodes = episodes
experiment_config.dqn.eval_interval = eval_interval
experiment_config.dqn.eval_games = eval_games
experiment_config.dqn.gradient_updates_per_step = 2

{
    "agendas": selected_agendas,
    "seeds": seeds,
    "episodes": experiment_config.dqn.episodes,
    "device": experiment_config.global_.device,
}


## Passo 2: Inspecao das agendas

Aqui verificamos que cada agenda segue a narrativa pretendida: random, weak, strong e self-play, com ou sem fase mista.


In [ ]:
for agenda_name in selected_agendas:
    definition = curricula[agenda_name]
    schedule, phase_summary = expand_curriculum_schedule(experiment_config.dqn.episodes, definition, seed=seeds[0])
    print(f"== {agenda_name} ===")
    print(definition.description)
    for item in phase_summary:
        print(item)
    counts = {}
    for phase in schedule:
        counts[phase.opponent_kind] = counts.get(phase.opponent_kind, 0) + 1
    print("counts:", counts)


## Passo 3: Execucao opcional dos treinos


In [ ]:
results = []

if run_training:
    for agenda_name in selected_agendas:
        for seed in seeds:
            run_config = copy.deepcopy(experiment_config)
            run_config.global_.seed = seed
            checkpoint_dir = CURRICULUM_OUTPUTS / f"{agenda_name}_seed_{seed}"
            print(f"=== Running {agenda_name} | seed={seed} ===")
            _agent, metrics = train_dqn_with_curriculum(
                curricula[agenda_name],
                run_config,
                checkpoint_dir=checkpoint_dir,
            )
            last_eval = metrics.evaluation[-1] if metrics.evaluation else {}
            print("Last eval:", last_eval)
            results.append(
                {
                    "agenda": agenda_name,
                    "seed": seed,
                    "best_score": metrics.best_score,
                    "last_eval": last_eval,
                    "best_checkpoint_path": metrics.best_checkpoint_path,
                }
            )
else:
    print("Training skipped. Set run_training = True to launch experiments.")

results[:2]


## Passo 4: Carregamento dos resultados guardados


In [ ]:
def load_curriculum_runs(root: Path) -> list[dict]:
    runs = []
    for metrics_path in sorted(root.glob("*_seed_*/metrics_final.json")):
        data = json.loads(metrics_path.read_text(encoding="utf-8"))
        run_name = metrics_path.parent.name
        family = run_name.split("_seed_")[0] if "_seed_" in run_name else run_name
        runs.append({
            "run_name": run_name,
            "family": family,
            "path": metrics_path,
            "data": data,
        })
    return runs

all_runs = load_curriculum_runs(CURRICULUM_OUTPUTS)
[(run["run_name"], len(run["data"].get("evaluation", []))) for run in all_runs]


## Passo 5: Agregacao por agenda


In [ ]:
def summarize_runs(runs: list[dict]) -> list[dict]:
    grouped: dict[str, list[dict]] = {}
    for run in runs:
        grouped.setdefault(run["family"], []).append(run)

    rows = []
    for family, family_runs in sorted(grouped.items()):
        best_scores = [float(run["data"].get("best_score", float("-inf"))) for run in family_runs]
        final_random = []
        final_weak = []
        final_strong = []
        final_previous = []
        for run in family_runs:
            evaluation = run["data"].get("evaluation", [])
            if not evaluation:
                continue
            last_eval = evaluation[-1]
            final_random.append(float(last_eval.get("vs_random_win_rate", 0.0)))
            final_weak.append(float(last_eval.get("vs_weak_heuristic_win_rate", 0.0)))
            final_strong.append(float(last_eval.get("vs_strong_heuristic_win_rate", 0.0)))
            final_previous.append(float(last_eval.get("vs_previous_win_rate", 0.0)))
        rows.append(
            {
                "agenda": family,
                "num_runs": len(family_runs),
                "mean_best_score": round(statistics.fmean(best_scores), 4) if best_scores else None,
                "mean_vs_random": round(statistics.fmean(final_random), 4) if final_random else None,
                "mean_vs_weak": round(statistics.fmean(final_weak), 4) if final_weak else None,
                "mean_vs_strong": round(statistics.fmean(final_strong), 4) if final_strong else None,
                "mean_vs_previous": round(statistics.fmean(final_previous), 4) if final_previous else None,
            }
        )
    return rows

summary_rows = summarize_runs(all_runs)
summary_rows


## Passo 6: Visualizacao rapida


In [ ]:
if summary_rows:
    agendas = [row["agenda"] for row in summary_rows]
    vs_random = [row["mean_vs_random"] or 0.0 for row in summary_rows]
    vs_weak = [row["mean_vs_weak"] or 0.0 for row in summary_rows]
    vs_strong = [row["mean_vs_strong"] or 0.0 for row in summary_rows]
    vs_previous = [row["mean_vs_previous"] or 0.0 for row in summary_rows]

    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    axes = axes.ravel()
    axes[0].bar(agendas, vs_random)
    axes[0].set_title("Final win rate vs random")
    axes[1].bar(agendas, vs_weak)
    axes[1].set_title("Final win rate vs weak")
    axes[2].bar(agendas, vs_strong)
    axes[2].set_title("Final win rate vs strong")
    axes[3].bar(agendas, vs_previous)
    axes[3].set_title("Final win rate vs previous snapshot")
    for ax in axes:
        ax.tick_params(axis="x", rotation=25)
    plt.tight_layout()
    plt.show()
else:
    print("No runs found yet.")


## Leituras esperadas

- se uma agenda sobe contra `weak` mas nao transfere para `strong`, o shaping pode estar a sobreajustar a padroes simples;
- se a fase `self_play` melhorar `vs_previous` mas nao `vs_strong`, o agente pode estar a explorar idiossincrasias da propria pool;
- se a agenda classica ganhar em `strong`, entao a narrativa random -> weak -> strong -> self-play faz sentido tambem no teu DQN.
